# Underwater Waste Detection with YOLO11l

YOLO object detection on **15 underwater-debris categories**.

**Training recorded in original Kaggle notebook:** 100 epochs at 640 × 640 resolution, on Tesla P100, with Ultralytics 8.3.237.

**Starting weights:** an existing detection checkpoint (`/kaggle/input/underwater/other/default/1/best.pt`), not a fresh YOLO11l base model.

> Original Kaggle notebooks are preserved separately. This is a cleaned portfolio copy made in November 2026. Historical numbers below come from saved outputs in the original notebooks; the cells in this copy have not been rerun.

This notebook has three parts: configuration, optional training, and validation/inference using an available trained `best.pt` file. Running it today creates new experiments; it does not recreate the original execution timestamp.

## Historical validation results

| Historical validation metric | Recorded result |
|---|---:|
| Precision | 0.850 |
| Recall | 0.750 |
| mAP@0.50 | 0.816 |
| mAP@0.50:0.95 | 0.534 |
| Validation images | 1,001 |
| Labeled instances | 1,891 |

The values above are transcribed from the **final validation of the saved `best.pt` checkpoint** in the original training output. They are **not** an independent test-set result and have **not** been reproduced by this cleaned notebook. The final excerpt is in `evidence/original_training_excerpt.txt`.

The original Kaggle notebook and its saved versions are retained privately; verify the original Kaggle version date there before claiming a specific publication date.

## 1. Install and load YOLO

Use a Kaggle GPU session if you intend to train. Attach the original dataset in the Kaggle **Input → Add Input** panel.

In [ ]:
%pip install -q ultralytics

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import yaml

## 2. Configuration

Leave `RUN_TRAINING=False` when you only want to evaluate a saved model. Set `SAVED_WEIGHTS` to the actual Kaggle input path of your archived `best.pt` checkpoint.

In [ ]:
DATA_YAML = Path("/kaggle/input/underwater-plastic-pollution-detection/underwater_plastics/data.yaml")
START_WEIGHTS = "/kaggle/input/underwater/other/default/1/best.pt"
EPOCHS = 100
IMAGE_SIZE = 640

RUN_TRAINING = False
SAVED_WEIGHTS = ""  # For example: /kaggle/input/my-model/other/default/1/best.pt

if not DATA_YAML.is_file():
    raise FileNotFoundError(f"Missing dataset YAML: {DATA_YAML}. Add the original dataset in Kaggle Input.")

with DATA_YAML.open(encoding="utf-8") as f:
    dataset = yaml.safe_load(f)

print("Dataset:", DATA_YAML)
print("Classes:", dataset.get("names", "not specified"))
print("Validation split:", dataset.get("val", "not specified"))

## 3. Training (optional)

The parameters below match the main original run. A new run will get a separate output directory. Do **not** label new results as the original 2025 run.

In [ ]:
trained_weights = None

if RUN_TRAINING:
    if START_WEIGHTS.endswith(".pt") and START_WEIGHTS != "yolo11l.pt" and not Path(START_WEIGHTS).is_file():
        raise FileNotFoundError(f"Missing starting checkpoint: {START_WEIGHTS}")

    model = YOLO(START_WEIGHTS)
    model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        project="/kaggle/working/runs/portfolio",
        name="reproduction",
        exist_ok=False,
    )
    trained_weights = Path(model.trainer.save_dir) / "weights" / "best.pt"
    print("Trained model:", trained_weights)
else:
    print("Training skipped. Set RUN_TRAINING=True to start a new experiment.")

## 4. Validate the trained checkpoint

Use the **validation** split, not the training split. If you only have the notebook but not the weights, recover `best.pt` from the original Kaggle version output or train a new model.

In [ ]:
weights_path = trained_weights or (Path(SAVED_WEIGHTS) if SAVED_WEIGHTS else None)

if weights_path is None:
    print("Validation skipped: set SAVED_WEIGHTS to your archived best.pt path, or run training above.")
elif not weights_path.is_file():
    raise FileNotFoundError(f"Model weights not found: {weights_path}")
else:
    model = YOLO(str(weights_path))
    metrics = model.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=IMAGE_SIZE,
        project="/kaggle/working/runs/portfolio",
        name="validation",
        exist_ok=False,
    )
    print(f"Precision: {metrics.box.mp:.3f}")
    print(f"Recall: {metrics.box.mr:.3f}")
    print(f"mAP@50: {metrics.box.map50:.3f}")
    print(f"mAP@50-95: {metrics.box.map:.3f}")

## 5. Preview predictions (optional)

Attach a licensed sample image or set the path to an image from your dataset. Check the image license before publishing examples.

In [ ]:
SAMPLE_IMAGE = ""  # Example: /kaggle/input/dataset/val/images/example.jpg

if not SAMPLE_IMAGE:
    print("Inference skipped: set SAMPLE_IMAGE to an image path to preview predictions.")
elif weights_path is None or not weights_path.is_file():
    print("Inference skipped: add trained weights first.")
elif not Path(SAMPLE_IMAGE).is_file():
    raise FileNotFoundError(f"Image not found: {SAMPLE_IMAGE}")
else:
    model = YOLO(str(weights_path))
    model.predict(
        source=SAMPLE_IMAGE,
        imgsz=IMAGE_SIZE,
        conf=0.25,
        save=True,
        project="/kaggle/working/runs/portfolio",
        name="predictions",
        exist_ok=False,
    )

## Notes

- The historical scores are **validation** results, not an external test or production benchmark.
- Dataset labels, data splits, initial checkpoint provenance, and potential dataset leakage should be independently reviewed before making stronger claims.
- The reconstructed notebook intentionally omits exploratory cells and failed experiments. The original notebook is unchanged and can be provided for verification.
- A fresh run may produce different results due to library versions, randomness, hardware, or data changes.